In [ ]:
# Colab bootstrap — auto-clone repo on Google Colab, no-op locally
import os, sys, subprocess

REPO = "https://github.com/jongmoonha/AI-PHM_Graduate.git"
DIR  = "AI-PHM_Graduate"

try:
    import google.colab  # type: ignore
    target = '/content/' + DIR
    if not os.path.isdir(target):
        subprocess.run(["git", "clone", REPO, target], check=True)
    os.chdir(target)
    print('Google Colab detected. Working directory:', os.getcwd())
except ImportError:
    print('Local environment detected. Working directory:', os.getcwd())


# Filtering Practice

시간영역 신호에서 원하는 주파수 대역만 추출하거나, 불필요한 저주파 드리프트 · 고주파 잡음을 제거하는 실습.

필터는 `utils.filtering_zerophase(x, fs, ftype, ...)` 한 함수로 적용한다. 내부적으로 `scipy.signal.butter` 로 설계한 Butterworth 필터를 `sosfiltfilt` (zero-phase) 로 앞뒤 두 번 통과시킨다. 인과(causal) 버전 `utils.filtering` 은 실습 2 의 위상 지연 비교에만 쓴다.

**Part A — 기본 개념** (3·10·15 Hz 사인파를 더한 합성 신호, 잡음 없음):
- 실습 1. 주파수 분석으로 신호 구성 확인하기
- 실습 2. Band-pass 필터로 관심 대역 추출하기 (causal vs zero-phase 비교)
- 실습 3. High-pass 필터로 저주파 사인파 제거
- 실습 4. Low-pass 필터로 고주파 사인파 제거
- 실습 5. High + Low ≈ Original (상보성 관찰)

**Part B — 응용 시나리오** (드리프트 + 공진 + 잡음):
- 실습 6. 응용 신호 — 드리프트 + 공진 + 잡음 (시간 파형 + 스펙트럼)
- 실습 7. HPF 로 드리프트 제거
- 실습 8. BPF 로 공진 대역 추출
- 실습 9. 신호 분해 — 드리프트 / 공진 / 잡음 3 성분 분리

## 준비: 라이브러리 & 데이터 로드

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from utils import fft, filtering, filtering_zerophase

In [ ]:
fs = 1000  # sampling frequency (Hz)

data = np.array(pd.read_csv('./data/data_sample_filtering.csv'))
print('shape:', data.shape)

t = data[:, 1]
v = data[:, 2]

---

## 실습 1. 주파수 분석으로 신호 구성 확인하기

필터를 설계하기 전에 **어떤 주파수 성분이 얼마나 섞여 있는가** 를 먼저 파악한다.
`utils.fft(v, fs)` 는 단측 진폭 스펙트럼 `(f, A)` 를 반환한다.

상단은 시간 영역, 하단은 0–30 Hz 를 확대한 주파수 영역이다. 시간 파형만으로는 몇 개의 성분이 섞였는지 세기 어렵지만, 스펙트럼에서는 성분의 개수와 주파수를 바로 읽을 수 있다.

In [ ]:
f, A = fft(v, fs)

fig, ax = plt.subplots(2, 1, figsize=(10, 6))
ax[0].plot(t, v, 'C0')
ax[0].set_xlabel('Time (s)')
ax[0].set_ylabel('y')
ax[0].set_title('Raw signal (time domain)')

ax[1].plot(f, A, 'C0')
ax[1].set_xlabel('Frequency (Hz)')
ax[1].set_ylabel('|Y|')
ax[1].set_xlim([0, 30])
ax[1].set_ylim([0, 1.2])
ax[1].set_title('Single-sided amplitude spectrum (0 to 30 Hz)')

fig.tight_layout()
plt.show()

**관찰.** 스펙트럼에는 3 Hz · 10 Hz · 15 Hz 세 자리에 진폭 1 의 피크가 나타나고 나머지 구간은 0 에 붙어 있다 — 이 신호는 서로 다른 주파수의 사인파 세 개로 이루어져 있고 잡음은 섞여 있지 않다. 차단 주파수를 5 Hz 로 잡으면 이 세 성분은 5 Hz 를 기준으로 저주파 쪽 1개 (3 Hz) 와 고주파 쪽 2개 (10 Hz · 15 Hz) 로 갈린다. 이후 실습에서는 이 경계를 이용해 원하는 성분만 골라내는 법을 확인한다.

---

## 실습 2. Band-pass 필터로 관심 대역 추출하기

`utils.filtering` 은 두 가지 방식을 제공한다.

- `filtering(x, fs, ftype, ...)` — **인과(causal) 단일 패스** (`sosfilt` 기반).
  실시간 시스템에 적용 가능하지만 **위상 지연** 이 누적된다.
- `filtering_zerophase(x, fs, ftype, ...)` — **영위상(zero-phase)** (`sosfiltfilt` 기반).
  신호를 앞뒤로 두 번 필터링해 위상 지연을 상쇄. 사후 분석 전용(비인과).
  크기 응답이 $|H|^2$ 가 되어 전이대가 더 가파르다.

두 방식을 같은 Band-pass 에 적용해 **위상 차이** 를 확인한다.

> **실무 팁**: 사후 분석에는 `filtering_zerophase` 를 기본으로 쓴다. 계수 차수가 높거나 차단 주파수가 Nyquist 에 매우 가까워 `filtering_zerophase` 가 수치적으로 발산하면 `filtering` (causal) 으로 우회한다. 이 노트북 이후의 모든 실습은 `filtering_zerophase` 를 사용한다.

In [ ]:
v_bp_causal = filtering(v,           fs, 'band', f_low=5, f_high=12)
v_bp_zero   = filtering_zerophase(v, fs, 'band', f_low=5, f_high=12)

fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax[0].plot(t, v,           color='C0', alpha=0.4, label='Original')
ax[0].plot(t, v_bp_causal, color='C1', ls='--', lw=1.2, label='Causal (sosfilt)')
ax[0].set_xlim([0, 2]); ax[0].set_ylabel('Amplitude'); ax[0].legend()
ax[0].set_title('Causal filter — real delay ~125 ms (more than one 10 Hz period)')

ax[1].plot(t, v,         color='C0', alpha=0.4, label='Original')
ax[1].plot(t, v_bp_zero, color='C2', ls='--', lw=1.2, label='Zero-phase (sosfiltfilt)')
ax[1].set_xlim([0, 2]); ax[1].set_xlabel('Time (s)'); ax[1].set_ylabel('Amplitude'); ax[1].legend()
ax[1].set_title('Zero-phase filter — no delay, peaks aligned with the 10 Hz component')

plt.tight_layout(); plt.show()

**관찰**:
- 상단 (causal `sosfilt`): 점선 (필터링 결과) 은 처음 약 0.2 초 동안 0 에서 서서히 커지고 0.5 초 무렵에야 일정한 모양에 이른다 — 출력이 입력보다 늦게 나온다는 흔적이다. 그 뒤 점선의 피크는 하단 점선 (원신호 속 10 Hz 성분과 같은 시각) 의 피크보다 약 25 ms 뒤에 선다. 그러나 이 25 ms 는 실제 지연이 아니다. 이 BPF 는 10 Hz 성분을 약 125 ms 늦추는데, 그 지연이 10 Hz 한 주기 (100 ms) 를 넘어서 피크끼리 맞대면 한 주기를 뺀 나머지 (약 25 ms) 만 보인다. 피크의 어긋남만으로 지연의 크기를 읽으면 안 된다.
- 하단 (zero-phase `sosfiltfilt`): 점선의 피크가 원신호 속 10 Hz 성분의 피크와 **같은 시각** (어긋남 5 ms 이내) 에 선다 — 원신호의 큰 피크 (0.42 · 1.42 초) 에 대 보면 바로 확인된다. 앞뒤 두 번 통과로 위상이 정확히 상쇄된다.
- 단, **신호 양 끝 경계** 는 `sosfiltfilt` 의 padding 처리 때문에 zero-phase 에서도 작은 왜곡이 남는다 — 이 그림에서는 0.02 초의 첫 피크가 다른 피크보다 조금 높다. 실무에서는 양 끝 일정 구간을 잘라내고 분석한다.
- 사후 분석에서는 "언제 일어났는가" 가 중요하므로 zero-phase 가 기본. 실시간 처리나 차단 주파수가 Nyquist 에 매우 가까워 `sosfiltfilt` 가 발산할 때만 causal 로 우회.

---

## 실습 3. High-pass 필터로 저주파 사인파 제거

차단 주파수 아래의 저주파 성분(DC · 드리프트 포함)을 제거하고 그 위 성분은 유지할 때 사용한다. 이 신호에서는 3 Hz 사인파가 제거 대상이다.

```python
utils.filtering_zerophase(x, fs, 'high', f_cut=...)
```

차단 주파수 5 Hz 로 필터링한 뒤 결과를 비교한다. 시간 파형은 앞 2 초만 확대해 그린다.

In [ ]:
cutoff = 5
v_hp = filtering_zerophase(v, fs, 'high', f_cut=cutoff)

f_raw, A_raw = fft(v, fs)
f_hp,  A_hp  = fft(v_hp, fs)

fig, ax = plt.subplots(2, 1, figsize=(10, 6))
ax[0].plot(t, v,    'C0', alpha=0.5, label='Original')
ax[0].plot(t, v_hp, 'C1',            label=f'Highpass (cutoff={cutoff} Hz)')
ax[0].set_xlim([0, 2])
ax[0].set_xlabel('Time (s)')
ax[0].set_ylabel('y')
ax[0].legend()

ax[1].plot(f_raw, A_raw, 'C0', alpha=0.5, label='Original')
ax[1].plot(f_hp,  A_hp,  'C1',            label='Filtered')
ax[1].set_xlabel('Frequency (Hz)')
ax[1].set_ylabel('|Y|')
ax[1].set_xlim([0, 30])
ax[1].set_ylim([0, 1.3])
ax[1].legend()

fig.tight_layout()
plt.show()

**관찰.** 스펙트럼에서 3 Hz 피크가 제거되고 10 Hz · 15 Hz 두 피크는 그대로 남는다. 시간 영역에서는 3 Hz 의 느린 출렁임이 사라지고 10 Hz 와 15 Hz 가 겹친 빠른 진동만 남는다. HPF 는 0 Hz 도 차단대역에 포함하므로 신호의 평균도 0 으로 맞춰진다.

---

## 실습 4. Low-pass 필터로 고주파 사인파 제거

차단 주파수 아래의 성분만 남기고 그 위 성분은 제거한다. 이 신호에서는 10 Hz · 15 Hz 사인파가 제거 대상이다.

```python
utils.filtering_zerophase(x, fs, 'low', f_cut=...)
```

실습 3 과 같은 차단 주파수 5 Hz 로 필터링해 상보적인 결과를 확인한다. 시간 파형은 앞 2 초만 확대해 그린다.

In [ ]:
cutoff = 5
v_lp = filtering_zerophase(v, fs, 'low', f_cut=cutoff)

f_raw, A_raw = fft(v, fs)
f_lp,  A_lp  = fft(v_lp, fs)

fig, ax = plt.subplots(2, 1, figsize=(10, 6))
ax[0].plot(t, v,    'C0', alpha=0.5, label='Original')
ax[0].plot(t, v_lp, 'C1',            label=f'Lowpass (cutoff={cutoff} Hz)')
ax[0].set_xlim([0, 2])
ax[0].set_xlabel('Time (s)')
ax[0].set_ylabel('y')
ax[0].legend()

ax[1].plot(f_raw, A_raw, 'C0', alpha=0.5, label='Original')
ax[1].plot(f_lp,  A_lp,  'C1',            label='Filtered')
ax[1].set_xlabel('Frequency (Hz)')
ax[1].set_ylabel('|Y|')
ax[1].set_xlim([0, 20])
ax[1].set_ylim([0, 1.3])
ax[1].legend()

fig.tight_layout()
plt.show()

**관찰.** 스펙트럼에서 10 Hz · 15 Hz 피크가 제거되고 3 Hz 피크만 남는다. 시간 파형은 3 Hz 단일 사인파로 매끈해진다. 실습 3 의 결과와 합치면 원신호가 복원될 것을 기대할 수 있다 — 이 관계를 다음 실습에서 확인한다.

---

## 실습 5. High + Low ≈ Original (상보성 관찰)

고역통과 + 저역통과 = 원신호 인지 시간 영역에서 검증한다. 영위상 필터 (`filtering_zerophase`) 를 사용한다.

In [ ]:
# HP + LP ≈ Original 확인 (zero-phase 필터)
v_hp = filtering_zerophase(v, fs, 'high', f_cut=5)
v_lp = filtering_zerophase(v, fs, 'low',  f_cut=5)

fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharey=True, gridspec_kw={'width_ratios': [2, 1]})
for a in ax:
    a.plot(t, v, color='C0', alpha=0.5, lw=1.5, label='Original')
    a.plot(t, v_hp + v_lp, color='C2', ls='--', lw=1.0, label='Recovered (v_hp + v_lp)')
    a.set_xlabel('Time (s)')
ax[0].set_xlim([0, 2]); ax[0].set_ylim([-3.4, 4.2]); ax[0].set_ylabel('Amplitude')
ax[0].legend(loc='upper center', ncol=2)
ax[0].set_title('First 2 s')
ax[1].set_xlim([9, 10])
ax[1].set_title('Last 1 s (boundary)')
fig.suptitle('Zero-phase HP + LP vs original')

plt.tight_layout(); plt.show()

**관찰**:
- 내부 구간에서 복구 신호 (점선) 가 원신호 (실선) 와 **거의 완벽히 일치** — $|H_{HP}(f)|^2 + |H_{LP}(f)|^2 = 1$ 성질 덕분.
- 앞 2 초 (왼쪽) 에서는 시작 경계까지 두 곡선이 완전히 겹친다. 오차는 신호 **끝 쪽** (오른쪽 패널, 약 9.8 초 이후 눈에 띄게) 에만 남아 마지막 샘플에서 약 1 에 이른다 — `sosfiltfilt` 의 padding 처리 때문이다. HP · LP 결과 각각에는 양 끝 모두 경계 왜곡이 있지만, 시작 쪽 왜곡은 더하면 서로 지워진다 — 이 신호에 한한 일이 아니라 `sosfiltfilt` 의 계산 방식에서 온다. 경계 오차가 양 끝에 똑같이 생긴다고 가정하지 말고, 실무에서는 양 끝을 적당히 잘라내고 분석한다.
- 완벽한 시간 영역 상보가 필요하면 `v_hp = v - v_lp` (감산) 같은 단순 방식이 대안.

---

## 실습 6. 응용 신호 — 드리프트 + 공진 + 잡음

여기부터 (실습 6 ~ 9) 는 **실제 측정 신호를 흉내 낸 시나리오** (드리프트 + 공진 + 잡음 합성) 위에서 필터링 워크플로를 시연한다.

새 데이터 `data_filtering_application.csv` 를 로드해 (변수 `v` 재할당) 가공 전 신호의 시간 파형과 스펙트럼을 본다. 데이터는 회전기계 가속도 신호를 모방한 합성 신호 — 저주파 드리프트 (0.5 Hz 진폭 1.5 + 0.3 Hz 진폭 0.5) + 6 ~ 10 Hz 광대역 공진 + 가우시안 잡음.

In [ ]:
data = np.array(pd.read_csv('./data/data_filtering_application.csv'))
t = data[:, 1]
v = data[:, 2]
print('shape:', data.shape)

f, A = fft(v, fs)

fig, ax = plt.subplots(2, 1, figsize=(10, 6))
ax[0].plot(t, v, 'C0', linewidth=0.6)
ax[0].set_xlabel('Time (s)'); ax[0].set_ylabel('y')
ax[0].set_title('Application signal: drift + resonance + noise')

ax[1].plot(f, A, 'C0')
ax[1].set_xlim([0, 30]); ax[1].set_ylim([0, 1.7])
ax[1].set_xlabel('Frequency (Hz)'); ax[1].set_ylabel('|Y|')
ax[1].set_title('Single-sided amplitude spectrum (0 to 30 Hz)')

fig.tight_layout(); plt.show()

**관찰.** 스펙트럼에서 세 성분이 확인된다 — (1) 0 ~ 2 Hz 의 드리프트 성분 (0.5 Hz 진폭 ≈ 1.5, 0.3 Hz 진폭 ≈ 0.5), (2) 6 ~ 10 Hz 공진 대역 (가장 높은 빈은 8.4 Hz 의 약 0.43, 대역 안 대부분은 0.1 ~ 0.35), (3) 그 밖의 평평한 잡음 바닥 (빈당 약 0.005 ~ 0.016). 잡음 바닥은 드리프트보다 100 배 가까이 작아 이 세로축에서는 0 에 붙은 선처럼 보인다. 이후 실습 7 ~ 9 는 이 세 성분을 단계별로 정리·분해한다.

---

## 실습 7. HPF 로 드리프트 제거

가장 먼저 제거할 것은 저주파 드리프트다. 차단 주파수 3 Hz 의 HPF 를 적용한다.

In [ ]:
v_hp = filtering_zerophase(v, fs, 'high', f_cut=3)

f_raw, A_raw = fft(v, fs)
f_hp,  A_hp  = fft(v_hp, fs)

fig, ax = plt.subplots(2, 1, figsize=(10, 6))
ax[0].plot(t, v,    'C0', alpha=0.5, lw=0.6, label='Original')
ax[0].plot(t, v_hp, 'C1', lw=0.6, label='Highpass (cutoff=3 Hz)')
ax[0].set_xlabel('Time (s)'); ax[0].set_ylabel('y'); ax[0].legend()

ax[1].plot(f_raw, A_raw, 'C0', alpha=0.5, label='Original')
ax[1].plot(f_hp,  A_hp,  'C1',            label='Filtered')
ax[1].set_xlim([0, 30]); ax[1].set_ylim([0, 1.7])
ax[1].set_xlabel('Frequency (Hz)'); ax[1].set_ylabel('|Y|'); ax[1].legend()

fig.tight_layout(); plt.show()

**관찰.** 시간 영역에서 큰 출렁임 (드리프트) 이 사라지고 빠른 진동 + 잡음만 남는다. 스펙트럼에서 0.3 · 0.5 Hz 드리프트 성분이 제거되고 6 ~ 10 Hz 공진 대역은 보존된다 — 드리프트가 제거된 신호 위에서 다음 단계 분석을 시작할 수 있다.

---

## 실습 8. BPF 로 공진 대역 추출

드리프트와 잡음을 동시에 제거하려면 BPF 한 줄로 충분하다. 5 ~ 12 Hz 통과대역을 잡아 공진 대역만 추출한다.

In [ ]:
v_bp = filtering_zerophase(v, fs, 'band', f_low=5, f_high=12)

f_raw, A_raw = fft(v, fs)
f_bp,  A_bp  = fft(v_bp, fs)

fig, ax = plt.subplots(2, 1, figsize=(10, 6))
ax[0].plot(t, v,    'C0', alpha=0.5, lw=0.6, label='Original')
ax[0].plot(t, v_bp, 'C2', lw=0.6, label='Bandpass (5-12 Hz)')
ax[0].set_xlabel('Time (s)'); ax[0].set_ylabel('y'); ax[0].legend()

ax[1].plot(f_raw, A_raw, 'C0', alpha=0.5, label='Original')
ax[1].plot(f_bp,  A_bp,  'C2',            label='Filtered')
ax[1].set_xlim([0, 30]); ax[1].set_ylim([0, 1.7])
ax[1].set_xlabel('Frequency (Hz)'); ax[1].set_ylabel('|Y|'); ax[1].legend()

fig.tight_layout(); plt.show()

**관찰.** 시간 영역에서 6~10 Hz 의 공진 진동만 남고, 스펙트럼에는 6~10 Hz 공진 대역만 남는다. 드리프트와 통과대역 밖의 잡음이 함께 제거되었다 — BPF 한 줄이 HPF + LPF 두 단계의 효과를 동시에 낸다.

---

## 실습 9. 신호 분해 — 3 성분 분리

같은 raw 신호로부터 드리프트 / 공진 / 잡음 세 성분을 모두 분리해 본다. LPF + BPF 두 줄과 한 줄의 산술이면 끝난다.

In [ ]:
drift     = filtering_zerophase(v, fs, 'low',  f_cut=3)
resonance = filtering_zerophase(v, fs, 'band', f_low=5, f_high=12)
noise     = v - drift - resonance

fig, ax = plt.subplots(4, 1, figsize=(11, 9), sharex=True)
ax[0].plot(t, v,         'C0', lw=0.6); ax[0].set_ylabel('y')
ax[0].set_title('(a) Raw signal (drift + resonance + noise)'); ax[0].set_ylim([-4, 4])

ax[1].plot(t, drift,     'C1', lw=1.0); ax[1].set_ylabel('y')
ax[1].set_title('(b) drift (LPF cutoff 3 Hz)'); ax[1].set_ylim([-4, 4])

ax[2].plot(t, resonance, 'C2', lw=0.6); ax[2].set_ylabel('y')
ax[2].set_title('(c) resonance (BPF 5-12 Hz)'); ax[2].set_ylim([-4, 4])

ax[3].plot(t, noise,     'C3', lw=0.6); ax[3].set_ylabel('y')
ax[3].set_xlabel('Time (s)')
ax[3].set_title('(d) residual noise (raw - drift - resonance)'); ax[3].set_ylim([-4, 4])

fig.tight_layout(); plt.show()

**관찰**:
- (b) 드리프트: 0.5 Hz·0.3 Hz 두 사인파의 합 (진폭 1.5·0.5, 출렁임 높이 1 ~ 2).
- (c) 공진: 빠른 진동 패킷, 진폭이 천천히 변하는 포락선 구조.
- (d) 잔여 잡음: 평균 0 부근의 가우시안 분포, 시간적 구조 없음 (백색 잡음). 양 끝 (처음 약 0.2 초 · 마지막 약 0.1 초) 의 출렁임은 LPF · BPF 의 경계 왜곡이다.

세 성분이 각각의 물리적 원인 (센서 기준선 변동 / 하우징 구조 공진 / 측정 잡음) 과 대응한다.

---